# Icarus — Hyperspectral Soil CNN Training
**HYPERVIEW2 dataset · Google Colab**

Before running: `Runtime → Change runtime type → T4 GPU`

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/icarus'

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive directory ready: {DRIVE_DIR}')

## 2. Clone repo

Opening this notebook from GitHub (via colab.research.google.com → File → Open → GitHub) only loads the `.ipynb`. This cell clones the rest of the repo (`train.py`, `model.py`, etc.) into the Colab session.

In [ ]:
import os

%cd /content

# Detect which repo this notebook was opened from
repo_url = None
try:
    # Colab sets this env var when a notebook is opened from GitHub
    colab_nb = os.environ.get('COLAB_NOTEBOOK_URL', '')
    if 'github.com' in colab_nb:
        parts = colab_nb.replace('https://github.com/', '').split('/')
        repo_url = f'https://github.com/{parts[0]}/{parts[1]}.git'
except Exception:
    pass

if repo_url is None:
    repo_url = 'https://github.com/davidshukhin/Icarus.git'  # fallback

print(f'Cloning: {repo_url}')
!git clone {repo_url} icarus
%cd /content/icarus

## 3. Install dependencies

In [ ]:
# Colab already has torch/numpy/scipy/sklearn — only install the extras
!pip install -q spectral rasterio pyyaml h5py
print('Dependencies installed.')

## 4. Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU — go to Runtime → Change runtime type → T4 GPU')

## 5. Config

In [ ]:
import os
import sys
sys.path.insert(0, '/content/icarus')

DRIVE_DIR  = '/content/drive/MyDrive/icarus'
CATALOG    = '/root/.cache/eotdl/datasets/HYPERVIEW2/catalog.v2.parquet'
DATA_DIR   = f'{DRIVE_DIR}/data/hyperview2'
SAVE_PATH  = f'{DRIVE_DIR}/best_model.pth'

os.makedirs(DATA_DIR, exist_ok=True)

CFG = {
    'model_name':        'se',
    'num_bands':         150,    # HYPERVIEW2 VNIR bands
    'num_classes':       5,      # soil health: 0 (degraded) → 4 (healthy)
    'num_contaminants':  4,
    'bottleneck_dim':    512,
    'dropout_p':         0.5,
    'target_size':       (64, 64),
    'batch_size':        32,
    'epochs':            50,
    'lr':                3e-4,
    'weight_decay':      0.05,
    'contam_loss_weight':0.5,
    'label_smoothing':   0.1,
    'cosine_T0':         10,
    'cosine_T_mult':     2,
    'grad_clip':         1.0,
    'num_workers':       2,
    'patience':          10,
    'save_path':         SAVE_PATH,
}

print('Config ready.')

## 6. Download HYPERVIEW2

`eotdl` fetches the STAC catalog. We then parse the catalog to get the actual patch file URLs and download them.

import requests
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

# ── Step 1: fetch STAC catalog via eotdl ─────────────────────────────────────
if not os.path.exists(CATALOG):
    !pip install -q eotdl
    !eotdl datasets get HYPERVIEW2 -v 2

# ── Step 2: parse catalog → (id, href) table ─────────────────────────────────
def parse_catalog(catalog_path):
    df = pd.read_parquet(catalog_path)
    records = []
    for _, row in df.iterrows():
        assets = row.get('assets', {})
        if isinstance(assets, dict):
            href = assets.get('asset', {}).get('href', '')
            records.append({'id': row['id'], 'href': href})
    return pd.DataFrame(records)

catalog = parse_catalog(CATALOG)
print(f'Catalog entries: {len(catalog)}')
print(catalog.head())

# ── Step 3: download all files (skips existing) ───────────────────────────────
def download_file(url, dest, chunk_size=8192):
    try:
        r = requests.get(url, stream=True, timeout=30)
        r.raise_for_status()
        with open(dest, 'wb') as f:
            for chunk in r.iter_content(chunk_size):
                f.write(chunk)
        return True
    except Exception as e:
        print(f'  Failed {Path(dest).name}: {e}')
        return False

skipped = downloaded = failed = 0
for _, row in tqdm(catalog.iterrows(), total=len(catalog), desc='Downloading'):
    dest = Path(DATA_DIR) / row['id']
    if dest.exists():
        skipped += 1
        continue
    dest.parent.mkdir(parents=True, exist_ok=True)
    if row['href'] and download_file(row['href'], dest):
        downloaded += 1
    else:
        failed += 1

print(f'\nDone — downloaded: {downloaded}, skipped: {skipped}, failed: {failed}')

## 7. Inspect labels & build health classes

import numpy as np
import matplotlib.pyplot as plt

LABELS_FILE = f'{DATA_DIR}/train_gt.csv'
gt = pd.read_csv(LABELS_FILE)

print(f'Patches: {len(gt)}')
print(f'Columns: {gt.columns.tolist()}')
print(gt.describe())

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, col in zip(axes, ['K', 'Mg', 'P2O5', 'pH']):
    if col in gt.columns:
        gt[col].hist(bins=30, ax=ax)
        ax.set_title(col)
plt.tight_layout()
plt.show()

# ── Map chemistry → health class 0–4 ─────────────────────────────────────────
_OPTIMA = {'K': (150., 250.), 'Mg': (100., 200.), 'P2O5': (40., 80.), 'pH': (6., 7.)}

def chemistry_to_health_class(df):
    scores = np.zeros(len(df), dtype=np.float32)
    n = 0
    for col, (lo, hi) in _OPTIMA.items():
        if col not in df.columns:
            continue
        vals = df[col].to_numpy(dtype=np.float32)
        s = np.ones_like(vals)
        s[vals < lo] = vals[vals < lo] / lo
        above = vals > hi
        s[above] = hi / np.maximum(vals[above], 1e-8)
        scores += s
        n += 1
    scores /= max(n, 1)
    thresholds = np.percentile(scores, [20, 40, 60, 80])
    classes = np.zeros(len(scores), dtype=int)
    for i, t in enumerate(thresholds):
        classes[scores > t] = i + 1
    return classes.tolist()

HEALTH_CLASS_LABELS = {0: 'Severely Degraded', 1: 'Degraded', 2: 'Moderate',
                        3: 'Recovering', 4: 'Healthy / Remediated'}

health_classes = chemistry_to_health_class(gt)
dist = np.bincount(health_classes)
print('\nHealth class distribution:')
for i, count in enumerate(dist):
    print(f'  {i} — {HEALTH_CLASS_LABELS[i]}: {count}')

## 8. Dataset

import torch
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import zoom

CONTAMINANT_NAMES = ['metal', 'pfas', 'glyphosate', 'microplastics']

class Hyperview2Dataset(Dataset):
    def __init__(self, data_dir, patch_ids, health_classes, num_bands=150,
                 target_size=(64, 64), train=True):
        self.data_dir      = Path(data_dir)
        self.patch_ids     = patch_ids
        self.health_classes = health_classes
        self.num_bands     = num_bands
        self.target_size   = target_size
        self.train         = train

    def _load_patch(self, patch_id):
        for ext in ('.npz', '.npy'):
            p = self.data_dir / f'{patch_id}{ext}'
            if p.exists():
                if ext == '.npz':
                    data = np.load(p)
                    cube = data[list(data.keys())[0]].astype(np.float32)
                else:
                    cube = np.load(p).astype(np.float32)
                # Ensure (C, H, W)
                if cube.ndim == 2:
                    cube = cube[np.newaxis]
                elif cube.ndim == 3 and cube.shape[2] < cube.shape[0]:
                    cube = cube.transpose(2, 0, 1)
                return cube
        raise FileNotFoundError(f'No patch file for {patch_id}')

    def __len__(self):
        return len(self.patch_ids)

    def __getitem__(self, idx):
        cube = self._load_patch(self.patch_ids[idx])

        # Resample bands & spatial size
        if cube.shape[0] != self.num_bands:
            cube = zoom(cube, (self.num_bands / cube.shape[0], 1, 1), order=1)
        tH, tW = self.target_size
        if (cube.shape[1], cube.shape[2]) != (tH, tW):
            cube = zoom(cube, (1, tH / cube.shape[1], tW / cube.shape[2]), order=1)

        # Random flips (training only)
        if self.train:
            if np.random.rand() > 0.5: cube = cube[:, ::-1, :].copy()
            if np.random.rand() > 0.5: cube = cube[:, :, ::-1].copy()

        # Band-wise z-score normalisation
        mean = cube.mean(axis=(1, 2), keepdims=True)
        std  = cube.std(axis=(1, 2),  keepdims=True) + 1e-8
        cube = (cube - mean) / std

        return (
            torch.from_numpy(cube).unsqueeze(0),                          # (1, C, H, W)
            torch.tensor(self.health_classes[idx], dtype=torch.long),
            torch.zeros(len(CONTAMINANT_NAMES), dtype=torch.float32),    # no contam GT
        )

# ── Build train / val split ───────────────────────────────────────────────────
id_col = next((c for c in gt.columns if 'id' in c.lower() or 'patch' in c.lower()), gt.columns[0])
patch_ids = gt[id_col].astype(str).tolist()

np.random.seed(42)
idx = np.random.permutation(len(patch_ids))
split = int(len(idx) * 0.8)
train_idx, val_idx = idx[:split], idx[split:]

train_ds = Hyperview2Dataset(DATA_DIR, [patch_ids[i] for i in train_idx],
                              [health_classes[i] for i in train_idx],
                              num_bands=CFG['num_bands'], target_size=CFG['target_size'], train=True)
val_ds   = Hyperview2Dataset(DATA_DIR, [patch_ids[i] for i in val_idx],
                              [health_classes[i] for i in val_idx],
                              num_bands=CFG['num_bands'], target_size=CFG['target_size'], train=False)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], pin_memory=torch.cuda.is_available(),
                          drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=torch.cuda.is_available())

print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

## 9. Model, loss & validation

In [ ]:
checkpoint = torch.load(SAVE_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Pick a sample from the val set
sample_idx = 0
cube_t, true_health, _ = val_ds[sample_idx]
patch_id = val_ds.patch_ids[sample_idx]

with torch.no_grad():
    ph, pc = model(cube_t.unsqueeze(0).to(device))

pred_class = ph.argmax(1).item()
confidence = ph.softmax(1).max().item()
contam_probs = pc.squeeze().cpu().numpy()

print(f'Patch:      {patch_id}')
print(f'True label: {HEALTH_CLASS_LABELS[val_ds.health_classes[sample_idx]]}')
print(f'Predicted:  {HEALTH_CLASS_LABELS[pred_class]} ({confidence:.1%} confidence)')
print()
print('Contaminant probabilities:')
for name, prob in zip(CONTAMINANT_NAMES, contam_probs):
    print(f'  {name}: {prob:.3f}')

## 12. Inference on a sample patch

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='train')
ax1.plot(history['val_loss'],   label='val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()

ax2.plot(history['val_acc'], label='accuracy')
ax2.plot(history['val_auc'], label='AUC')
ax2.set_title('Validation'); ax2.set_xlabel('Epoch'); ax2.legend()

plt.tight_layout()
plt.show()

## 11. Results

In [ ]:
from tqdm.notebook import tqdm as tqdm_nb

history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}
best_auc      = -1.0
patience_left = CFG['patience']

for epoch in range(CFG['epochs']):
    model.train()
    running_loss = 0.0
    n = 0

    pbar = tqdm_nb(enumerate(train_loader), total=len(train_loader),
                   desc=f"Epoch {epoch:03d}", leave=False)

    for i, (cube, health, contam) in pbar:
        cube, health, contam = cube.to(device), health.to(device), contam.to(device)
        optimizer.zero_grad()

        with torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu'):
            ph, pc = model(cube)
            loss, _, _ = compute_loss(ph, pc, health, contam)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step(epoch + i / len(train_loader))

        running_loss += loss.item(); n += 1
        pbar.set_postfix(loss=f'{running_loss/n:.4f}', lr=f"{optimizer.param_groups[0]['lr']:.2e}")

    metrics = validate(model, val_loader)
    history['train_loss'].append(running_loss / n)
    history['val_loss'].append(metrics['val_loss'])
    history['val_acc'].append(metrics['val_acc'])
    history['val_auc'].append(metrics['val_auc'])

    print(f"Epoch {epoch:03d} | train {running_loss/n:.4f} | "
          f"val loss {metrics['val_loss']:.4f}  acc {metrics['val_acc']:.3f}  AUC {metrics['val_auc']:.3f}")

    if metrics['val_auc'] > best_auc:
        best_auc = metrics['val_auc']
        patience_left = CFG['patience']
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'best_auc': best_auc, 'cfg': CFG}, SAVE_PATH)
        print(f"  Saved best model → {SAVE_PATH}")
    else:
        patience_left -= 1
        if patience_left == 0:
            print(f'Early stopping at epoch {epoch}.')
            break

print(f'\nBest val AUC: {best_auc:.4f}')

## 10. Train

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.metrics import roc_auc_score
from model import create_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = create_model(
    CFG['model_name'],
    num_bands=CFG['num_bands'],
    num_classes=CFG['num_classes'],
    num_contaminants=CFG['num_contaminants'],
    bottleneck_dim=CFG['bottleneck_dim'],
    dropout_p=CFG['dropout_p'],
).to(device)
print(f"Model: {CFG['model_name']}  params: {sum(p.numel() for p in model.parameters()):,}")

optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=CFG['cosine_T0'], T_mult=CFG['cosine_T_mult'])
scaler    = torch.amp.GradScaler('cuda' if torch.cuda.is_available() else 'cpu')
ce        = nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])

def compute_loss(ph, pc, health, contam):
    ce_loss  = ce(ph, health)
    bce_loss = F.binary_cross_entropy(pc, contam)
    return ce_loss + CFG['contam_loss_weight'] * bce_loss, ce_loss, bce_loss

@torch.no_grad()
def validate(model, loader):
    model.eval()
    total_loss = correct = total = n = 0
    all_probs, all_labels = [], []
    for cube, health, contam in loader:
        cube, health, contam = cube.to(device), health.to(device), contam.to(device)
        ph, pc = model(cube)
        loss, _, _ = compute_loss(ph, pc, health, contam)
        total_loss += loss.item(); n += 1
        correct += (ph.argmax(1) == health).sum().item()
        total   += health.size(0)
        all_probs.append(pc.cpu().numpy())
        all_labels.append(contam.cpu().numpy())

    import numpy as np
    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    auc_scores = []
    for c in range(labels.shape[1]):
        if labels[:, c].sum() > 0 and (1 - labels[:, c]).sum() > 0:
            auc_scores.append(roc_auc_score(labels[:, c], probs[:, c]))

    return {
        'val_loss': total_loss / n,
        'val_acc':  correct / total,
        'val_auc':  float(np.mean(auc_scores)) if auc_scores else float('nan'),
    }

print('Model and helpers ready.')